This is a notebook for developing a method for batch processing CRISM VRDR products and resampling them to CaSSIS wavelengths, by accessing an external storage device where the VRDR dataset is hosted. 

We will also resample to PanCam and OROCHI wavelengths. We could also do Mastcam, Mastcam-Z, HiRISE, HRSC whilst we're here.

# Overview

1. Access the External Storage, and list the quadrants and tiles to process
2. Make or access a master list of processed tiles
2. Access a VRDR Tile
3. Access the CaSSIS transmission profiles (or other profiles)
4. Apply the transmission profiles to the VRDR tile
5. Save the CaSSIS band simulated image, to GeoTIFF, ENVI, and if possible PDS .IMG formats

# Dependencies

In [ ]:
from pathlib import Path
import geopandas as gpd
from osgeo import gdal
from pyproj import CRS, Transformer
import rasterio as rio
from rasterio.transform import Affine
import spectral.io.envi as envi
from collections import OrderedDict
import pdr

# 1. Accessing the VRDR External Storage

## Connect to the External Drive

In [ ]:
ext_drive_name = 'RS_STORE'

In [ ]:
ext_drive_path = Path('/Volumes', ext_drive_name)
print('External Drive found: ', ext_drive_path.exists())

## Find the VRDR directory

In [ ]:
vrdr_dir = Path(ext_drive_path, 'vrdr')
print('VRDR Directory found: ', vrdr_dir.exists())

## Get the list of quadrants

In [ ]:
# get the list of directories in vrdr_dir that start with mc
mc_dirs = [d for d in vrdr_dir.iterdir() if d.is_dir() and d.name.startswith('mc')]
print('MC directories found: ', len(mc_dirs))
for d in mc_dirs:
    print(' - ', d.name)

## Get the list of products in a Quadrant

In [ ]:
mc_dir = mc_dirs[0]
print('Processing MC Directory: ', mc_dir.name)

In [ ]:
# get lbl files but ignore hidden files
mc_lbl_files = [f for f in mc_dir.glob('*.lbl') if not f.name.startswith('.')]
# sort
mc_lbl_files.sort()
print('Number of LBL files found: ', len(mc_lbl_files))

In [ ]:
for lbl in mc_lbl_files:
    print(' - ', lbl.stem)

## TODO: Make Shared Progress Log File

# 3. Access a VRDR Tile and Load

In [ ]:
lbl_path = mc_lbl_files[0]

In [ ]:
vrral = pdr.read(lbl_path)
band_names = vrral.metaget('BAND_NAME')
crism_vrdr_wvls = [float(band) for band in band_names]

# 3. Access the Instrument Transmission Profiles

In [ ]:
# read the CaSSIS-2019.csv file
import pandas as pd

def load_channel_profiles(instrument: str) -> pd.DataFrame:
    """Load the channel information from the instrument CSV file."""
    inst_csv_path = Path('.', instrument).with_suffix('.csv')
    if not inst_csv_path.exists():
        raise FileNotFoundError(f"{instrument} CSV file not found: {inst_csv_path}")
    
    inst_df = pd.read_csv(inst_csv_path)

    inst_df = inst_df.set_index('wvls')

    return inst_df

In [ ]:
cassis_df = load_channel_profiles('CaSSIS-2019')

In [ ]:
cassis_df.plot()

# 4. Apply the Transmission Profiles to the VRDR Tile

In [ ]:
from typing import Literal
import numpy as np
import time
from scipy.interpolate import interp1d

def apply_band(
        band: str, 
        inst_df: pd.DataFrame,
        crism_vrdr_wvls: list, 
        crism_vrdr_cube: np.ndarray) -> np.ndarray:
    # get the selected cassis band transmission profile
    band_profile = inst_df[band].values
    profile_wvls = inst_df.index

    interp_func = interp1d(profile_wvls, band_profile, kind='linear', bounds_error=False, fill_value='extrapolate')
    resampled_profile = interp_func(crism_vrdr_wvls)
    # clip
    resampled_profile = np.clip(resampled_profile, 0, None)

    # set max values to nan
    crism_vrdr_cube[crism_vrdr_cube == crism_vrdr_cube.max()] = np.nan

    # efficiently mulitply the CRISM VRDR cube by the resampled CaSSIS profile
    band_cube = crism_vrdr_cube * resampled_profile[:, np.newaxis, np.newaxis]
    # efficiently sum the cube along the spectral axis, and divide by the sum of the resampled profile to normalise
    band_image = np.nansum(band_cube, axis=0) / np.sum(resampled_profile)

    return band_image

In [ ]:
def apply_bands_2_cube(
        band_arr: np.ndarray,
        band_wvls: np.ndarray,
        cube: np.ndarray,
        cube_wvls: np.ndarray,
        no_data: float
        ) -> np.ndarray:
    """Apply multiple band profiles to a cube."""
    interp_funcs = [interp1d(band_wvls, band_profile.flatten(), kind='linear', bounds_error=False, fill_value='extrapolate') for band_profile in band_arr.T]
    resampled_profiles = np.array([interp_func(cube_wvls) for interp_func in interp_funcs])
    # clip
    resampled_profiles = np.clip(resampled_profiles, 0, None)

    # set max values to nan
    cube[cube == no_data] = np.nan

    band_cube = []
    for band in resampled_profiles:        
        band_image = np.sum(cube * band[:, np.newaxis, np.newaxis], axis=0) / np.sum(band)
        band_cube.append(band_image)

    band_cube = np.nan_to_num(band_cube, nan=no_data)

    return np.array(band_cube)

In [ ]:
vrral.load('IMAGE', reload=True)
crism_vrdr_cube = vrral.IMAGE
no_data = vrral.metaget('IMAGE')['MISSING_CONSTANT']

In [ ]:
band_list = cassis_df.columns.tolist()
band_arr = cassis_df.to_numpy()
band_wvls = cassis_df.index.to_numpy()
band_cwls = np.round(np.sum(band_arr*band_wvls[:, np.newaxis], axis=0) / np.sum(band_arr, axis=0))

In [ ]:
cassis_cube = apply_bands_2_cube(band_arr, band_wvls, crism_vrdr_cube, np.array(crism_vrdr_wvls), no_data)

In [ ]:
def apply_stretch(image: np.ndarray, lower_percentile: float=0.01, upper_percentile: float=99.95, no_data=no_data) -> np.ndarray:
    """Apply a linear stretch to the image based on the given percentiles."""
    # treat no_data values as NaN
    image = image.astype(float)
    stretch_img = image.copy()
    stretch_img[image == no_data] = np.nan
    lower = np.nanpercentile(stretch_img, lower_percentile)
    upper = np.nanpercentile(stretch_img, upper_percentile)
    print(f"Stretching with lower={lower:.4f}, upper={upper:.4f}")
    stretched = (stretch_img - lower) / (upper - lower)
    stretched = np.clip(stretched, 0.0, 1.0)

    # convert to 8-bit
    stretched = (stretched * 254).astype(np.uint8)
    stretched[np.isnan(stretch_img)] = 255
    
    return stretched

In [ ]:
cassis_stretch_cube = []
for band in cassis_cube:
    band_stretch = apply_stretch(band)
    cassis_stretch_cube.append(band_stretch)
cassis_stretch_cube = np.array(cassis_stretch_cube)

In [ ]:
# make browse product
def cassis_browse_product(
        product: Literal['NPB', 'NRB', 'NRP', 'RPB', 'RGB'],
        cassis_cube,
        cassis_stretch_cube,
        no_data,
    ) -> np.array:

    if product == 'NPB':
        band_list = ['NIR', 'PAN', 'BLU']
        browse_arr = np.array([cassis_stretch_cube[3], cassis_stretch_cube[1], cassis_stretch_cube[0]])

    if product == 'NRB':
        band_list = ['NIR', 'RED', 'BLU']
        browse_arr = np.array([cassis_stretch_cube[3], cassis_stretch_cube[2], cassis_stretch_cube[0]])

    if product == 'NRP':
        band_list = ['NIR', 'RED', 'PAN']
        browse_arr = np.array([cassis_stretch_cube[3], cassis_stretch_cube[2], cassis_stretch_cube[1]])
    
    if product == 'RPB':
        band_list = ['RED', 'PAN', 'BLU']
        browse_arr = np.array([cassis_stretch_cube[2], cassis_stretch_cube[1], cassis_stretch_cube[0]])

    if product == 'RGB':
        band_list = ['PAN', 'BLU', 'sBLU']
        # make synthetic blu channel
        # mask with no data
        no_data_mask = (cassis_cube[0] != no_data) & (cassis_cube[1] != no_data)
        sBLU = (2*cassis_cube[0]) - (0.3*cassis_cube[1])
        sBLU[~no_data_mask] = no_data
        sBLU_stretch = apply_stretch(sBLU, no_data=no_data)
        browse_arr = np.array([cassis_stretch_cube[1], cassis_stretch_cube[0], sBLU_stretch])

    browse_arr = browse_arr.transpose((1, 2, 0))

    return browse_arr, band_list

In [ ]:
from typing import Tuple

def cassis_spectral_parameters(
        parameter: Literal[
            'PBR', #PAN/BLU
            'PNR', #PAN/NIR
            'RPR', #RED/PAN
            'ICE', #MEAN[B,P,R,N]/(NIR/BLU)
            'CBRC1', #(RPR, PBR, PNR)->(R,G,B)
            'CBRC2', #(RPR, PBR, ICE)->(R,G,B)
        ],
        cassis_cube,
        no_data        
    )->Tuple[np.array,np.array]:
    
    no_data_locs = cassis_cube == no_data

    bands=1
    if parameter == 'PBR':
        band_list = ['PBR']
        sp_arr = np.divide(cassis_cube[1], cassis_cube[0], where=~no_data_locs[0])

    if parameter == 'PNR':
        band_list = ['PNR']
        sp_arr = np.divide(cassis_cube[1], cassis_cube[3], where=~no_data_locs[3])

    if parameter == 'RPR':
        band_list = ['RPR']
        sp_arr = np.divide(cassis_cube[2], cassis_cube[1], where=~no_data_locs[1])
    
    if parameter == 'ICE':
        band_list = ['ICE']
        sp_arr = np.divide(np.mean(cassis_cube, axis=0), (cassis_cube[3] / cassis_cube[0]), where=~no_data_locs[0])
    
    if parameter == 'CBRC1':
        bands = 3
        band_list = ['RPR', 'PBR', 'PNR']
        sp_arr = np.array([
            np.divide(cassis_cube[2], cassis_cube[1], where=~no_data_locs[1]),
            np.divide(cassis_cube[1], cassis_cube[0], where=~no_data_locs[0]),
            np.divide(cassis_cube[1], cassis_cube[3], where=~no_data_locs[3])
        ])
    
    if parameter == 'CBRC2':
        bands = 3
        band_list = ['RPR', 'PBR', 'ICE']
        sp_arr = np.array([
            np.divide(cassis_cube[2], cassis_cube[1], where=~no_data_locs[1]),
            np.divide(cassis_cube[1], cassis_cube[0], where=~no_data_locs[0]),
            np.divide(np.mean(cassis_cube, axis=0), (cassis_cube[3] / cassis_cube[0]), where=~no_data_locs[0])
        ])

    # for each sp_arr channel, apply_stretch
    if bands == 1:
        sp_stretch = apply_stretch(sp_arr, no_data=no_data)
    if bands == 3:
        sp_stretch = np.stack([apply_stretch(sp_arr[band], no_data=no_data) for band in range(bands)], axis=0)
        sp_stretch = sp_stretch.transpose((1, 2, 0))
    

    return sp_arr, sp_stretch, band_list


In [ ]:
def summary_product_stack(
        cassis_cube: np.ndarray,
        no_data: float
    ) -> np.array:
    """Create a spectral parameter stack"""
    sp_list = ['PBR','PNR','RPR','ICE']
    sp_cube = []
    sp_stretch_cube = []
    sp_band_list = []
    for sp_id in sp_list:
        sp_data, sp_stretch, sp_bands = cassis_spectral_parameters(sp_id, cassis_cube, no_data)
        sp_cube.append(sp_data)
        sp_stretch_cube.append(sp_stretch)
        sp_band_list.append(sp_bands[0])
    sp_cube = np.stack(sp_cube, axis=2)
    sp_stretch_cube = np.stack(sp_stretch_cube, axis=2)
    return sp_cube, sp_stretch_cube, sp_band_list

In [ ]:
npb, npb_bands = cassis_browse_product('NPB', cassis_cube, cassis_stretch_cube, no_data)

In [ ]:
rgb, rgb_bands = cassis_browse_product('RGB', cassis_cube, cassis_stretch_cube, no_data)

In [ ]:
csups_stack, csups_stretch_stack, csups_band_list = summary_product_stack(cassis_cube, no_data)

# cbrc1
cbrc1,cbrc1_stretch, cbrc1_bands  = cassis_spectral_parameters('CBRC1', cassis_cube, no_data)

In [ ]:
from matplotlib import pyplot as plt

plt.imshow(cbrc1_stretch)

# 5. Export the CaSSIS Simulated Tile

We need to export the CaSSIS cube as a georeferenced GeoTIFF.

This means we need to get the CRS information from the VRDR PDS cube, and then parse this to a new raster object, that we write both the cube and the CRS information too.

In [ ]:
cassis_vrdr_path = vrdr_dir.parent / 'vrdr_cassis'
cassis_vrdr_path.mkdir(exist_ok=True)   

In [ ]:
out_dir = cassis_vrdr_path / mc_dir.stem
out_dir.mkdir(exist_ok=True, parents=True)

In [ ]:
def export_geotiff(
        cassis_cube: np.ndarray,
        no_data: float,
        band_list: list,
        band_cwls: list,
        lbl_path: Path,
        out_dir: Path
    ) -> Path:
    """Export the cassis cube as a GeoTIFF file with georeferencing from the VRDR LBL file."""
    pds_file = lbl_path.with_suffix('.img')

    cube_stem = lbl_path.stem.replace('vrral', 'ccube')

    out_path = out_dir / (cube_stem + '.tif')

    pds_gdal = gdal.Open(str(pds_file))
    pds_transform = Affine.from_gdal(*pds_gdal.GetGeoTransform())
    pds_crs = CRS.from_wkt(pds_gdal.GetProjection())
    arr = cassis_cube.transpose((1, 2, 0))
    arr.shape
    with rio.open(out_path, 'w', driver='GTiff',
                                height = arr.shape[0], width = arr.shape[1],
                                count=4, dtype=str(arr.dtype),
                                compress='LZW',
                                nodata=no_data,
                                crs=pds_crs, transform=pds_transform) as new_dataset:
        for band_idx, band in enumerate(band_list):
            new_dataset.write(arr[:,:,band_idx], band_idx + 1)    
            new_dataset.set_band_description(band_idx+1, band)
            new_dataset.update_tags(band_idx+1, WAVELENGTH=f'{band_cwls[band_idx]}nm')
    new_dataset.close()
    
    return out_path

In [ ]:
def export_browse_product(
        browse_product_data: np.ndarray,
        browse_product_id: str,
        no_data: int,
        band_list: list,
        lbl_path: Path,
        out_dir: Path
    ) -> Path:
    """Export a cassis browse product as a GeoTIFF file with georeferencing from the VRDR LBL file."""
    pds_file = lbl_path.with_suffix('.img')

    # replace vrral with  cb{browse_product_id}
    cb_browse_product_id = f'cb{browse_product_id.lower()}'
    browse_product_stem = lbl_path.stem.replace('vrral', cb_browse_product_id)

    out_path = out_dir / (browse_product_stem + '.tif')

    pds_gdal = gdal.Open(str(pds_file))
    pds_transform = Affine.from_gdal(*pds_gdal.GetGeoTransform())
    pds_crs = CRS.from_wkt(pds_gdal.GetProjection())    
    arr = browse_product_data    

    # if 1 band, expand dims duplicate band 3 times    
    if len(arr.shape) == 2:
        arr = np.expand_dims(arr, axis=2)
        arr = np.repeat(arr, 3, axis=2)
        # expand band list by duplication
        band_list = band_list * 3

    with rio.open(out_path, 'w', driver='GTiff',
                                height = arr.shape[0], width = arr.shape[1],
                                count=3, dtype=str(arr.dtype),
                                compress='LZW',
                                nodata=no_data,
                                crs=pds_crs, transform=pds_transform) as new_dataset:
        for band_idx, band in enumerate(band_list):
            new_dataset.write(arr[:,:,band_idx], band_idx + 1)    
            new_dataset.set_band_description(band_idx+1, band)            
    new_dataset.close()
    
    return out_path

In [ ]:
def export_sp_stack(
        sp_stack: np.ndarray,
        no_data: int,
        sp_ids: list,
        lbl_path: Path,
        out_dir: Path
    ) -> Path:
    """Export a cassis browse product as a GeoTIFF file with georeferencing from the VRDR LBL file."""
    pds_file = lbl_path.with_suffix('.img')

    # replace vrral with csups
    csups_stem = lbl_path.stem.replace('vrral', 'csups')
    out_path = out_dir / (csups_stem + '.tif')

    pds_gdal = gdal.Open(str(pds_file))
    pds_transform = Affine.from_gdal(*pds_gdal.GetGeoTransform())
    pds_crs = CRS.from_wkt(pds_gdal.GetProjection())    
    arr = sp_stack
    with rio.open(out_path, 'w', driver='GTiff',
                                height = arr.shape[0], width = arr.shape[1],
                                count=len(sp_ids), dtype=str(arr.dtype),
                                compress='LZW',
                                nodata=no_data,
                                crs=pds_crs, transform=pds_transform) as new_dataset:
        for sp_idx, sp in enumerate(sp_ids):
            new_dataset.write(arr[:,:,sp_idx], sp_idx + 1)    
            new_dataset.set_band_description(sp_idx+1, sp)
    new_dataset.close()
    
    return out_path

In [ ]:
def write_cube_envi_hdr(lbl_path, out_dir, band_list, band_cwls)->Path:
    envi_header_path = lbl_path.with_suffix('.hdr')
    # read in the envi header as a text file into a dictionary
    with open(envi_header_path, 'r') as f:
        envi_header_text = f.readlines()
    envi_header_dict = OrderedDict()
    for line in envi_header_text:
        if '=' in line:
            key, value = line.split('=', 1)
            if key.strip() == 'band names':
                # get the number of bands
                n_bands = int(envi_header_dict['bands'])  
                envi_header_dict['band names'] = ''
                continue
            if key.strip() == 'wavelength':
                # get the number of bands
                n_bands = int(envi_header_dict['bands'])  
                envi_header_dict['wavelength'] = ''
                continue
            envi_header_dict[key.strip()] = value.strip()

    envi_header_dict['bands'] = str(len(band_list))
    envi_header_dict['band names'] = ', '.join(band_list)
    envi_header_dict['wavelength'] = ', '.join([f'{wvl}' for wvl in band_cwls])
    envi_header_dict['wavelength']
    envi_header_dict['file type'] = 'TIFF'
    output_envi_hdr_path = out_dir / envi_header_path.name

    ccube_stem = lbl_path.stem.replace('vrral', 'ccube')    
    output_envi_hdr_path = (out_dir / ccube_stem).with_suffix('.hdr')

    # write the dict to a text file, as a new envi header file
    with open(output_envi_hdr_path, 'w') as f:
        f.write('ENVI\n')
        for key, value in envi_header_dict.items():
            if key in ['band names', 'wavelength']:
                f.write(f'{key} = {{')
                # write the values in lines of max 70 characters
                line = ''
                for item in value.split(', '):
                    if len(line) + len(item) + 2 > 70:
                        f.write(line[:-2] + '\n')
                        line = ''
                    line += item + ', '
                f.write(line[:-2] + '}\n')
            else:
                f.write(f'{key} = {value}\n')
    return output_envi_hdr_path

In [ ]:
def write_sups_envi_hdr(lbl_path, out_dir, sp_list, default_bands)->Path:
    envi_header_path = lbl_path.with_suffix('.hdr')
    # read in the envi header as a text file into a dictionary
    with open(envi_header_path, 'r') as f:
        envi_header_text = f.readlines()
    envi_header_dict = OrderedDict()
    for line in envi_header_text:
        if '=' in line:
            key, value = line.split('=', 1)
            if key.strip() == 'band names':
                # get the number of bands
                n_bands = int(envi_header_dict['bands'])  
                envi_header_dict['band names'] = ''
                continue
            if key.strip() == 'wavelength':
                # get the number of bands
                n_bands = int(envi_header_dict['bands'])  
                envi_header_dict['wavelength'] = ''
                continue
            envi_header_dict[key.strip()] = value.strip()

    envi_header_dict['bands'] = str(len(sp_list))
    envi_header_dict['band names'] = ', '.join(sp_list)
    # remove 'wavelength' entry
    if 'wavelength' in envi_header_dict:
        del envi_header_dict['wavelength']
    envi_header_dict['file type'] = 'TIFF'
    envi_header_dict['default bands'] = ', '.join(str(x) for x in default_bands)

    # replace vrral with csups
    csups_stem = lbl_path.stem.replace('vrral', 'csups')    
    output_envi_hdr_path = (out_dir / csups_stem).with_suffix('.hdr')

    # write the dict to a text file, as a new envi header file
    with open(output_envi_hdr_path, 'w') as f:
        f.write('ENVI\n')
        for key, value in envi_header_dict.items():
            if key in ['band names', 'default bands']:
                f.write(f'{key} = {{')
                # write the values in lines of max 70 characters
                line = ''
                for item in value.split(', '):
                    if len(line) + len(item) + 2 > 70:
                        f.write(line[:-2] + '\n')
                        line = ''
                    line += item + ', '
                f.write(line[:-2] + '}\n')
            else:
                f.write(f'{key} = {value}\n')
    return output_envi_hdr_path

# 6. Batch Process for all Tiles

In [ ]:
mc_dirs

In [ ]:
['mc11', 'mc13','mc29']

In [ ]:
cassis_df = load_channel_profiles('CaSSIS-2019')
band_list = cassis_df.columns.tolist()
band_arr = cassis_df.to_numpy()
band_wvls = cassis_df.index.to_numpy()
band_cwls = np.round(np.sum(band_arr*band_wvls[:, np.newaxis], axis=0) / np.sum(band_arr, axis=0))

cassis_vrdr_path = vrdr_dir.parent / 'vrdr_cassis'
cassis_vrdr_path.mkdir(exist_ok=True)  

for mc_dir in mc_dirs:
    if mc_dir.stem not in ['mc11', 'mc13','mc29']:
        continue
    print('Processing MC Directory: ', mc_dir.name)
    # get lbl files but ignore hidden files
    mc_lbl_files = [f for f in mc_dir.glob('*.lbl') if not f.name.startswith('.')]
    # sort
    mc_lbl_files.sort()
    print('Number of LBL files found: ', len(mc_lbl_files))
    for lbl_path in mc_lbl_files:
        print(' - ', lbl_path.stem)

        out_dir = cassis_vrdr_path / mc_dir.stem / lbl_path.stem
        if out_dir.exists():
            print('Output directory exists, skipping: ', out_dir)
            continue
        
        out_dir.mkdir(exist_ok=True, parents=True)

        vrral = pdr.read(lbl_path)
        band_names = vrral.metaget('BAND_NAME')
        crism_vrdr_wvls = [float(band) for band in band_names]

        vrral.load('IMAGE', reload=True)
        crism_vrdr_cube = vrral.IMAGE
        no_data = vrral.metaget('IMAGE')['MISSING_CONSTANT']

        cassis_cube = apply_bands_2_cube(band_arr, band_wvls, crism_vrdr_cube, np.array(crism_vrdr_wvls), no_data)

        cassis_stretch_cube = []
        for band in cassis_cube:
            print('Cube:')
            band_stretch = apply_stretch(band, no_data=no_data)
            cassis_stretch_cube.append(band_stretch)
        cassis_stretch_cube = np.array(cassis_stretch_cube)

        for bp_id in ['NPB', 'NRB', 'NRP', 'RPB', 'RGB']:
            print('Browse Products')
            bp_data, bp_bands = cassis_browse_product(bp_id, cassis_cube, cassis_stretch_cube, no_data)
            export_browse_product(bp_data, bp_id, 255, bp_bands, lbl_path, out_dir)

        print('Summary Products')
        csups_stack, csups_stretch_stack, csups_band_list = summary_product_stack(cassis_cube, no_data)
        export_sp_stack(csups_stack, no_data,csups_band_list,lbl_path, out_dir)
        write_sups_envi_hdr(lbl_path, out_dir, csups_band_list, default_bands=[3, 1, 2])

        print('Summary Browse Products')
        # cbrc1
        cbrc1,cbrc1_stretch, cbrc1_bands  = cassis_spectral_parameters('CBRC1', cassis_cube, no_data)
        export_browse_product(cbrc1_stretch, 'CBRC1', 255, cbrc1_bands, lbl_path, out_dir)
        # cbrc2
        cbrc2,cbrc2_stretch, cbrc2_bands  = cassis_spectral_parameters('CBRC2', cassis_cube, no_data)
        export_browse_product(cbrc2_stretch, 'CBRC2', 255, cbrc2_bands, lbl_path, out_dir)

        out_file = export_geotiff(
            cassis_cube,
            no_data,
            band_list,
            band_cwls,
            lbl_path,
            out_dir
        )

        out_hdr = write_cube_envi_hdr(lbl_path, out_dir, band_list, band_cwls)